# 02 — Depth: years of M30 (TwelveData chunked + Stooq CSV)

Academic research only; not financial advice. The 8-month panel from notebook 01 is too short for walk-forward folds. This notebook builds a **multi-year M30 panel** from two $0 sources:

- **TwelveData free** — the vetted, dated source. The refresh is chunked (`max_partitions`) and resumable (stage ledger), so you can grow history year by year without hitting free-tier caps in one run.
- **Stooq** — Stooq now blocks automated downloads with a JS bot-wall, so the honest flow is: print the URL, download the CSV in a browser, upload it here, and import it through the same provenance pipeline.

Prereqs: run 00_BOOTSTRAP first, and store `OMEGA_TWELVEDATA_API_KEY` in secrets.


In [ ]:
import os, sys
PROJECT = '/content/omega_worldclass_engine'
if not os.path.isdir(PROJECT):
    PROJECT = '/kaggle/working/omega_worldclass_engine'
sys.path.insert(0, PROJECT)
os.chdir(PROJECT)
print('project:', PROJECT)
# Repair Kaggle's broken base numpy/scipy before importing omega.
import subprocess, sys
subprocess.run([sys.executable, 'scripts/repair_scientific_stack.py'], check=True)


In [ ]:
# Pull latest code and confirm the provider key is present.
!git -C {PROJECT} pull --quiet
from omega.secrets import load_platform_secrets
status = load_platform_secrets()
print('twelvedata_key_present:', status['OMEGA_TWELVEDATA_API_KEY'])


## Step 1 — Grow the panel backward, one chunk at a time

Pick a target start (e.g. 2022-01-01) and an end that is the first day of the current month. `refresh_dataset` only fetches months that are missing or incomplete, so re-running the cell appends the next `CHUNK` months. Repeat until `already_present` covers the window, then the next cell can load the full panel.


In [ ]:
from omega.cloud_config import load_cloud_config
from omega.acquisition import refresh_dataset

cloud = load_cloud_config('config/cloud_twelvedata.yaml')
# Target window. Month-aligned; end = first day of the current month.
START = '2022-01-01T00:00:00Z'
END = '2026-09-01T00:00:00Z'
# Months fetched per run (free-tier friendly). Increase after quotas recover.
CHUNK = 6

refresh = refresh_dataset(
    cloud, PROJECT, START, END,
    accept_provider_terms=True,
    max_partitions=CHUNK,
)
print('fetched:', refresh['fetched_partitions'])
print('already present:', refresh['already_present'])
print('skipped (next run):', refresh['skipped_partitions'])


## Step 2 — Optional: import a Stooq CSV you downloaded in a browser

1. In a browser, open the URL printed below and save the CSV.
2. Upload it here (folder icon in the left panel → upload).
3. Set `STOOQ_CSV` to its path and run this cell.

Stooq timestamps are exchange-local and naive, so `--timezone` is required. Use UTC unless you verified the pair's Stooq timezone.


In [ ]:
from import_stooq import stooq_url
print('Download in a browser:', stooq_url('eurusd', 30, '2024-01-01', '2026-06-01'))


In [ ]:
STOOQ_CSV = '/content/stooq_eurusd.csv'  # path to your uploaded CSV
STOOQ_TZ = 'UTC'
if os.path.isfile(STOOQ_CSV):
    !python scripts/import_stooq.py {STOOQ_CSV} --instrument EUR_USD --start 2024-01-01 --end 2026-06-01 --timezone {STOOQ_TZ}
else:
    print('No Stooq CSV found at', STOOQ_CSV, '- skipping.')


## Step 3 — Load the full multi-year panel and train

`load_dataset` fails closed if any requested month is missing or incomplete, so a partial history can never silently enter training. Only run this once `already_present` above covered the whole window.


In [ ]:
from omega.acquisition import load_dataset
from omega.config import load_config
from omega.pipeline import run_pipeline

panel = load_dataset(cloud, PROJECT, START, END)
print('rows:', len(panel), '| range:', panel.timestamp.min(), '->', panel.timestamp.max())

research = load_config('config.yaml')
research['data_source'] = cloud['data_source']
metrics = run_pipeline(panel, research, 'artifacts/depth_run')
display(metrics.groupby(['label', 'model'])[['brier', 'average_precision', 'ece']].mean().round(4))
